# Read repair

Reads every object with consistency level `ALL`, which makes Weaviate compare all replicas of the object and repair any that are out of date.

Use it after replicas may have drifted, e.g. after a node outage, on collections without async replication. With async replication enabled, replicas are repaired in the background and this is rarely needed.

Multi-tenant collections are repaired tenant by tenant; only HOT (active) tenants can be read.

Connection settings are read from a `.env` file next to the notebook (or from environment variables). The connection cell lists them.

In [ ]:
%pip install -U weaviate-client python-dotenv

In [ ]:
import os

import httpx
import weaviate
from dotenv import load_dotenv
from weaviate.classes.init import AdditionalConfig, Auth, Timeout

# Settings come from a .env file (or environment variables):
#   WEAVIATE_HOST       host without http:// or https://   (default: localhost)
#   WEAVIATE_HTTP_PORT  default 8080                       (443 for Weaviate Cloud)
#   WEAVIATE_GRPC_HOST  default: same as WEAVIATE_HOST     (grpc-<host> for Weaviate Cloud)
#   WEAVIATE_GRPC_PORT  default 50051                      (443 for Weaviate Cloud)
#   WEAVIATE_SECURE     true for HTTPS/TLS                 (true for Weaviate Cloud)
#   WEAVIATE_API_KEY    leave unset for anonymous access
# Weaviate Cloud can also use weaviate.connect_to_weaviate_cloud(cluster_url=..., auth_credentials=Auth.api_key(...)).
load_dotenv(".env", override=True)  # .env next to the notebook

HOST = os.getenv("WEAVIATE_HOST", "localhost")
HTTP_PORT = int(os.getenv("WEAVIATE_HTTP_PORT", "8080"))
GRPC_HOST = os.getenv("WEAVIATE_GRPC_HOST") or HOST
GRPC_PORT = int(os.getenv("WEAVIATE_GRPC_PORT", "50051"))
SECURE = os.getenv("WEAVIATE_SECURE", "false").lower() == "true"
API_KEY = os.getenv("WEAVIATE_API_KEY")

client = weaviate.connect_to_custom(
    http_host=HOST,
    http_port=HTTP_PORT,
    http_secure=SECURE,
    grpc_host=GRPC_HOST,
    grpc_port=GRPC_PORT,
    grpc_secure=SECURE,
    auth_credentials=Auth.api_key(API_KEY) if API_KEY else None,
    additional_config=AdditionalConfig(timeout=Timeout(init=30, query=60, insert=120)),
)

# The raw schema, exactly as the server reports it
response = httpx.get(
    f"{'https' if SECURE else 'http'}://{HOST}:{HTTP_PORT}/v1/schema",
    headers={"Authorization": f"Bearer {API_KEY}"} if API_KEY else {},
    timeout=30,
)
response.raise_for_status()
SCHEMA = {c["class"]: c for c in response.json().get("classes") or []}

print(f"Weaviate {client.get_meta()['version']} (client {weaviate.__version__})")
print(f"Ready: {client.is_ready()} | Live: {client.is_live()} | Connected: {client.is_connected()}")
print(f"Collections: {len(SCHEMA)}")

In [ ]:
# --- Read repair ---
from weaviate.classes.config import ConsistencyLevel
from weaviate.classes.tenants import TenantActivityStatus

COLLECTIONS = None  # e.g. ["Articles"]; None repairs every collection
LOG_EVERY = 10_000  # progress line every N objects

HOT_STATUSES = (TenantActivityStatus.ACTIVE, TenantActivityStatus.HOT)


def repair(collection, label):
    """Read each object with consistency ALL; return the number of objects that could not be read."""
    strict = collection.with_consistency_level(ConsistencyLevel.ALL)
    checked = errors = 0
    for obj in collection.iterator(return_properties=[]):
        try:
            if strict.query.fetch_object_by_id(obj.uuid) is None:
                print(f"  {label} | {obj.uuid} | not found on all replicas")
                errors += 1
        except Exception as e:
            print(f"  {label} | {obj.uuid} | {e}")
            errors += 1
        checked += 1
        if checked % LOG_EVERY == 0:
            print(f"  {label}: {checked} objects checked")
    print(f"{label}: {checked} objects checked, {errors} errors")
    return errors


total_errors = 0
for name in COLLECTIONS or sorted(SCHEMA):
    collection = client.collections.use(name)
    if not (SCHEMA[name].get("multiTenancyConfig") or {}).get("enabled"):
        total_errors += repair(collection, name)
        continue
    for tenant, info in collection.tenants.get().items():
        if info.activity_status in HOT_STATUSES:
            total_errors += repair(collection.with_tenant(tenant), f"{name}/{tenant}")
        else:
            print(f"{name}/{tenant}: tenant is {info.activity_status.value}, skipped (only HOT tenants can be read)")

print(f"\nDone, {total_errors} errors.")

In [ ]:
client.close()